# Create and solve low-resolution surrogates
This notebook constructs and solves the low-resolution surrogates, to enable model reduction.  

**Author:** Zhi Gao  
**Affiliation:** Utrecht University  
**Date:** June 2026  

**License:** MIT  
This notebook is provided for academic reproducibility only.  
Please cite the associated publication if you use this code.

## Table of Contents
1. Model configuration
1. Import pacakages & data loading
1. Initialize variable sets and lower- and upper-bounds
1. Create and solve underestimation
1. Update bounds and remove variables from underestimation
1. Create and solve overestimation
1. Update bounds and remove variables from overestimation
1. Save variable sets, bounds and metrics

In [ ]:
# ============================================================
# Scenario Definition
# ============================================================
YEAR = 1985 # or 2014

# Structural assumptions
WITH_STORAGE = True
WITH_TRANSMISSION = True

# Operational extensions
WITH_GRID_TARIFF = True
WITH_STORAGE_DISSIPATION = True

# ============================================================
# Surrogate Resolution Mapping
# ============================================================
SURROGATE_RESOLUTION = 'seasonal' # or any of below

dict_n_periods = {
    # String key -> number of clusters
    "seasonal": 4,      # 4 clusters per year
    "monthly": 12,       # 12 clusters per year
    "weekly": 59,        # ~59 weeks per year
    "daily": 365,        # 365 days per year
    "8h": 365 * 3,       # 3 clusters per day (8h each)
    "4h": 365 * 6,       # 6 clusters per day (4h each)
    "2h": 365 * 12,      # 12 clusters per day (2h each)
}

N_CLUSTERS = dict_n_periods[SURROGATE_RESOLUTION]

print(f"Scenario: Year={YEAR}, "
      f"Storage={WITH_STORAGE}, "
      f"Transmission={WITH_TRANSMISSION}, "
      f"Tariff={WITH_GRID_TARIFF}, "
      f"Dissipation={WITH_STORAGE_DISSIPATION}, "
      f"Surrogate resolution={SURROGATE_RESOLUTION}")

In [ ]:
# ============================================================
# Import packages
# ============================================================

# Optimization
import pyomo.environ as pyo

# Data handling
import pandas as pd
import numpy as np

# System & timing
import timeit
import os

# Utilities
from utils.solver_utils import get_rss_GB
from utils.surrogate_resolution import build_time_clusters
import pickle
# ============================================================
# Data Loading and Preprocessing
# ============================================================

# -------- Technology capacity --------
df_tech_capacity = pd.read_csv(
    "data/tech_capacity.csv",
    header=0, index_col=0
)

df_line_capacity = pd.read_csv(
    "data/line_capacity.csv",
    header=0, index_col=0
)

# -------- Apply scenario flags --------
if not WITH_STORAGE:
    df_tech_capacity["Battery"] *= 0
    df_tech_capacity["Battery_MWh"] *= 0
    df_tech_capacity["Hydro_Storage"] *= 0
    df_tech_capacity["Hydro_Storage_GWh"] *= 0
    
if not WITH_TRANSMISSION:
    df_line_capacity *= 0

# -------- Grid tariff --------
if WITH_GRID_TARIFF:
    df_grid_tarif = pd.read_csv(
        "data/grid_tarif.csv",
        index_col=0
    )

# ============================================================
# Technology Parameters
# ============================================================

# Storage characteristics
if WITH_STORAGE_DISSIPATION:
    dict_storage_hourly_self_dissipate_rate = {
        "Battery": 0.0001,
        "Hydro_Storage": 0.0001
    }

dict_storage_cycle_efficiency = {
    "Battery": 0.955,
    "Hydro_Storage": 0.875,
}

# Conversion efficiency
dict_conversion_efficiency = {
    "PEMEC": 0.585,
    "PEMFC": 0.5,
}

# Variable cost (Euro/MWh)
dict_variable_cost = {
    "Wind_Onshore": 2.06,
    "Wind_Offshore": 4.17,
    "Solar": 0.0,
    "CCGT": 79.0,
    "Battery": 0.1,
    "OCGT": 109.93,
    "Coal": 51.0,
    "Nuclear": 11.5,
    "E_ENS": 1000.0,
    "Run_of_River": 0.2,
    "Hydro_Storage": 0.2,
}

# ============================================================
# Sets Construction
# ============================================================

# Countries
list_country = np.unique(
    [asset[0:2] for asset in df_tech_capacity.index.values]
)

# Lines with positive capacity
list_line = [
    line for line in df_line_capacity.index.values
    if df_line_capacity.loc[line].values[0] > 0
]

# Country–technology output pairs
list_country_tech_output = []
for country in list_country:
    list_country_tech_output.append((country, "E_ENS"))
    for tech in [
        "Solar", "Wind_Offshore", "Wind_Onshore",
        "Nuclear", "Battery", "Coal", "CCGT",
        "OCGT", "Run_of_River", "Hydro_Storage"
    ]:
        if df_tech_capacity.loc[country, tech] > 0:
            list_country_tech_output.append((country, tech))

# Country–technology input (charging) pairs
list_country_tech_input = []
for country in list_country:
    for tech in ["Battery", "Hydro_Storage"]:
        if df_tech_capacity.loc[country, tech] > 0:
            list_country_tech_input.append((country, tech))

# ============================================================
# Time Series Profiles
# ============================================================

demand_profile = pd.read_csv(
    f"data/profiles/demand_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

solar_profile = pd.read_csv(
    f"data/profiles/solar_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

wind_offshore_profile = pd.read_csv(
    f"data/profiles/offshore_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

wind_onshore_profile = pd.read_csv(
    f"data/profiles/onshore_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

run_of_river_profile = pd.read_csv(
    f"data/profiles/run_of_river_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

dict_tech_profile = {
    "Solar": solar_profile,
    "Wind_Offshore": wind_offshore_profile,
    "Wind_Onshore": wind_onshore_profile,
    "Run_of_River": run_of_river_profile,
}


print("✅ Data and sets successfully prepared.")

In [ ]:
# ============================================================
# Surrogate Models Initialization
# ============================================================

list_producer_tech = ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River','Nuclear','Coal','CCGT','OCGT']

# -------- Precompute country → tech mapping --------
country_to_techs = {}
for country in list_country:
    country_to_techs[country] = [
        tech for (c, tech) in list_country_tech_output if c == country
    ]

# -------- Initialize variable bound tracking --------
dict_vary_tech_output = {}
dict_capacity_tech_output = {}
dict_vary_tech_output_lower_bound = {}
dict_vary_tech_output_upper_bound = {}

print("Initializing variable bound tracking...")

# Precompute renewable techs for faster lookup
renewable_techs = {"Wind_Onshore", "Wind_Offshore", "Solar", "Run_of_River"}

for country in list_country:
    techs = country_to_techs[country]
    
    for hour in range(1, 8761):
        # Use tuple keys for faster hashing
        key = (country, hour)
        
        dict_vary_tech_output[key] = techs.copy()
        dict_capacity_tech_output[key] = []
        
        for tech in techs:
            dict_vary_tech_output_lower_bound[(country, tech, hour)] = 0
            
            if tech in renewable_techs:
                profile = dict_tech_profile[tech]
                upper = profile.loc[hour, country]
            elif tech in ["Battery", "Nuclear", "Coal", "CCGT", "OCGT", "Hydro_Storage"]:
                upper = df_tech_capacity.loc[country, tech]
            elif tech == "E_ENS":
                upper = demand_profile[country].max()
            else:
                continue
                
            dict_vary_tech_output_upper_bound[(country, tech, hour)] = upper

print("✅ Variable bound tracking initialized")

In [ ]:
# ============================================================
# Build Underestimation Model
# ============================================================

# Time clustering
list_timestep = build_time_clusters(N_CLUSTERS)

# Country-tech-timestep combinations
list_country_tech_timestep_output = [
    (country, tech, t)
    for country, tech in list_country_tech_output
    for t in range(len(list_timestep))
    if tech in dict_vary_tech_output[country, list_timestep[t][0]]
]

print(f"🔄 Solving UNDERESTIMATION")
print(f"   Time clusters: {len(list_timestep)}")
print(f"   Variables: {len(list_country_tech_timestep_output)}")

start_time = timeit.default_timer()

# -------- Model definition --------
model_under = pyo.ConcreteModel()

model_under.set_country = pyo.Set(initialize=list_country)
model_under.set_line = pyo.Set(initialize=list_line)
model_under.set_timestep = pyo.Set(initialize=range(len(list_timestep)))
model_under.set_country_tech_input = pyo.Set(
    initialize=list_country_tech_input
)
model_under.set_country_tech_timestep_output = pyo.Set(
    initialize=list_country_tech_timestep_output
)

# Variables
model_under.var_transmission = pyo.Var(
    model_under.set_line,
    model_under.set_timestep,
    within=pyo.NonNegativeReals,
)

model_under.var_tech_output = pyo.Var(
    model_under.set_country_tech_timestep_output,
    within=pyo.NonNegativeReals,
)

model_under.var_tech_input = pyo.Var(
    model_under.set_country_tech_input,
    model_under.set_timestep,
    within=pyo.NonNegativeReals,
)

model_under.var_stored_energy = pyo.Var(
    model_under.set_country_tech_input,
    model_under.set_timestep,
    within=pyo.NonNegativeReals,
)

# -------- Constraints --------
def energy_balance_under(m, country, t):
    hours = list_timestep[t]
    # Compute output for generators operating at full capacity
    if len(dict_capacity_tech_output[country, hours[0]]) > 0:
        capacity_output_renewable = 0 * demand_profile.loc[hours, country]
        for tech in dict_capacity_tech_output[country, hours[0]]:
            if tech in ["Wind_Onshore", "Wind_Offshore", "Solar", "Run_of_River"]:
                capacity_output_renewable += dict_tech_profile[tech].loc[
                    hours, country
                ]
            else:
                capacity_output_renewable += df_tech_capacity.loc[
                    country, tech
                ] * np.ones_like(demand_profile.loc[hours, country])
    else:
        capacity_output_renewable = 0 * demand_profile.loc[hours, country]

    residual_demand = demand_profile.loc[hours, country] - capacity_output_renewable

    return (
        0.9 * pyo.quicksum(
            m.var_transmission[line, t]
            for line in m.set_line
            if line[3:5] == country
        )
        - pyo.quicksum(
            m.var_transmission[line, t]
            for line in m.set_line
            if line[0:2] == country
        )
        + pyo.quicksum(
            m.var_tech_output[country, tech, t]
            for tech in dict_vary_tech_output[country, hours[0]]
        )
        - pyo.quicksum(
            m.var_tech_input[country_tech, t]
            for country_tech in list_country_tech_input
            if country_tech[0] == country
        )
        == residual_demand.min()
    )

model_under.constraint_energy_balance = pyo.Constraint(
    model_under.set_country,
    model_under.set_timestep,
    rule=energy_balance_under,
)

def transmission_capacity_under(m, line, t):
    return m.var_transmission[line, t] <= df_line_capacity.loc[line, "Capacity"]

model_under.constraint_transmission_capacity = pyo.Constraint(
    model_under.set_line,
    model_under.set_timestep,
    rule=transmission_capacity_under,
)

def tech_output_capacity_under(m, country, tech, t):
    hours = list_timestep[t]
    if tech in ["Wind_Onshore", "Wind_Offshore", "Solar", "Run_of_River"]:
        return m.var_tech_output[country, tech, t] <= (
            dict_tech_profile[tech].loc[hours, country].max() # Taking the maximum of renewable available power
        )
    elif tech in [
        "Battery", "Nuclear", "Coal", "CCGT", "OCGT", "Hydro_Storage"
    ]:
        return m.var_tech_output[country, tech, t] <= (
            df_tech_capacity.loc[country, tech]
        )
    elif tech == "E_ENS":
        return m.var_tech_output[country, tech, t] <= (
            demand_profile[country].max()
        )
    else:
        raise ValueError(f"Unknown tech: {tech}")

model_under.constraint_tech_output_capacity = pyo.Constraint(
    model_under.set_country_tech_timestep_output,
    rule=tech_output_capacity_under,
)

def tech_input_capacity(model_under,country,tech,timestep):
    name_capacity = tech
    return model_under.var_tech_input[country,tech,timestep] <= df_tech_capacity.loc[country,name_capacity]
model_under.constraint_tech_input_capacity = pyo.Constraint(model_under.set_country_tech_input,model_under.set_timestep,\
                                                            rule=tech_input_capacity)

def stored_energy_capacity_uninvestable(model_under,country,tech,timestep):
    if tech == 'Battery':
        return model_under.var_stored_energy[country,tech,timestep] <= df_tech_capacity.loc[country,'Battery_MWh']
    else:
        return model_under.var_stored_energy[country,tech,timestep] <= 1000 * df_tech_capacity.loc[country,tech+'_GWh']
model_under.constraint_stored_energy_capacity_uninvestable = pyo.Constraint(model_under.set_country_tech_input,model_under.set_timestep,rule=stored_energy_capacity_uninvestable)


def storage_balance_under(m, country, tech, t):
    hours = list_timestep[t]
    eff = dict_storage_cycle_efficiency[tech]

    if WITH_STORAGE_DISSIPATION:
        diss = dict_storage_hourly_self_dissipate_rate[tech]
    else:
        diss = 0.0

    if t == 0:
        return (
            m.var_stored_energy[country, tech, t]
            - (1 - diss)**len(hours) * m.var_stored_energy[country, tech, len(list_timestep) - 1]
            == len(hours) * (
                m.var_tech_input[country, tech, t]
                - (1 / eff) * m.var_tech_output[country, tech, t]
            )
        )
    else:
        return (
            m.var_stored_energy[country, tech, t]
            - (1 - diss)**len(hours) * m.var_stored_energy[country, tech, t - 1]
            == len(hours) * (
                m.var_tech_input[country, tech, t]
                - (1 / eff) * m.var_tech_output[country, tech, t]
            )
        )

model_under.constraint_storage_balance = pyo.Constraint(
    model_under.set_country_tech_input,
    model_under.set_timestep,
    rule=storage_balance_under,
)

# -------- Objective --------
def total_cost_under(m):
    gen_cost = pyo.quicksum(
        len(list_timestep[t]) * dict_variable_cost[tech] *
        m.var_tech_output[country, tech, t]
        for (country, tech, t) in m.set_country_tech_timestep_output
    )

    if WITH_GRID_TARIFF:
        trans_cost = pyo.quicksum(
            df_grid_tarif.loc[line].values[0] *
            m.var_transmission[line, t]
            for line in m.set_line
            for t in m.set_timestep
        )
    else:
        trans_cost = 0

    return gen_cost + trans_cost

model_under.obj = pyo.Objective(rule=total_cost_under, sense=pyo.minimize)

# -------- Solve --------
solver = pyo.SolverFactory("gurobi")
model_under.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

mem_before = get_rss_GB()
solution_under = solver.solve(model_under, tee=True)
mem_after = get_rss_GB()

wall_time = timeit.default_timer() - start_time

total_time_under = wall_time
solver_time_under = float(solution_under["Solver"][0]["Wall time"])
objective_under = float(solution_under["Problem"][0]["Lower bound"])
under_delta_memory = mem_after - mem_before

print(f"✅ Underestimation solved in {wall_time:.2f}s")
print(f"   Objective: {objective_under:,.2f}")

In [ ]:
# ============================================================
# Extract Bounds from Underestimation
# ============================================================

dual_constraint_capacity_output_under = {}
output_tech_under = {}
removed_hours_under = 0

for country, tech in list_country_tech_output:
    if tech in list_producer_tech:
        output_tech_under[country, tech] = []
        dual_constraint_capacity_output_under[country, tech] = []

        for t in range(len(list_timestep)):
            if tech in dict_vary_tech_output[country, list_timestep[t][0]]:
                val = pyo.value(model_under.var_tech_output[country, tech, t])
                output_tech_under[country, tech].append(val > 1e-5)

                upper_bound = max(
                    dict_vary_tech_output_upper_bound[
                        country, tech, h
                    ]
                    for h in list_timestep[t]
                )
                dual_constraint_capacity_output_under[country, tech].append(
                    abs(upper_bound - val) < 1e-5
                )
            else:
                output_tech_under[country, tech].append(False)
                dual_constraint_capacity_output_under[country, tech].append(
                    False
                )

# Update variable bounds
for country, tech in list_country_tech_output:
    if tech in list_producer_tech:
        for t in range(len(list_timestep)):
            if tech in dict_vary_tech_output[country, list_timestep[t][0]]:
                if dual_constraint_capacity_output_under[country, tech][t]:
                    for hour in list_timestep[t]:
                        removed_hours_under += 1
                        dict_capacity_tech_output[country, hour].append(
                            tech
                        )
                        dict_vary_tech_output[country, hour].remove(
                                tech
                        )
                else:
                    for hour in list_timestep[t]:
                        dict_vary_tech_output_lower_bound[
                            country, tech, hour
                        ] = pyo.value(
                            model_under.var_tech_output[country, tech, t]
                        )

reduced_variables_under = removed_hours_under

print(f"✅ Bounds extracted")
print(f"Underestimation removed variables: {removed_hours_under:,}")

In [ ]:
# ============================================================
# Surrogate Overestimation Model 
# ============================================================

print(f"🔄 Solving OVERESTIMATION")
# Solve overestimation problem
list_country_tech_timestep_output = [(country,tech,timestep) for country,tech in list_country_tech_output for timestep in range(len(list_timestep)) if tech in dict_vary_tech_output[country,list_timestep[timestep][0]]]
start_time = timeit.default_timer()

model_over_bounded = pyo.ConcreteModel()
model_over_bounded.set_country = pyo.Set(initialize=list_country)
model_over_bounded.set_line = pyo.Set(initialize=list_line)
model_over_bounded.set_timestep = pyo.Set(initialize=range(len(list_timestep)))
model_over_bounded.set_country_tech_timestep_output = pyo.Set(initialize=list_country_tech_timestep_output)
model_over_bounded.set_country_tech_input = pyo.Set(initialize=list_country_tech_input)

model_over_bounded.var_transmission = pyo.Var(model_over_bounded.set_line,model_over_bounded.set_timestep,within=pyo.NonNegativeReals)

# Technology for every country
model_over_bounded.var_tech_output = pyo.Var(model_over_bounded.set_country_tech_timestep_output,within=pyo.NonNegativeReals)
model_over_bounded.var_tech_input = pyo.Var(model_over_bounded.set_country_tech_input,model_over_bounded.set_timestep,within=pyo.NonNegativeReals)
model_over_bounded.var_stored_energy = pyo.Var(model_over_bounded.set_country_tech_input,model_over_bounded.set_timestep,within=pyo.NonNegativeReals)

def energy_balance_electricity(model_over_bounded,country,timestep):
    if len(dict_capacity_tech_output[country,list_timestep[timestep][0]]):
        capacity_output_renewable = 0 * demand_profile.loc[list_timestep[timestep],country]
        for tech in dict_capacity_tech_output[country,list_timestep[timestep][0]]:
            if tech in ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River']:
                capacity_output_renewable += dict_tech_profile[tech].loc[list_timestep[timestep],country]
            else:
                capacity_output_renewable += df_tech_capacity.loc[country,tech] * np.ones_like(demand_profile.loc[list_timestep[timestep],country])
    else:
        capacity_output_renewable = 0 * demand_profile.loc[list_timestep[timestep],country]
    
    residual_demand = demand_profile.loc[list_timestep[timestep],country] - capacity_output_renewable

    return 0.9 * pyo.quicksum(model_over_bounded.var_transmission[line,timestep] for line in model_over_bounded.set_line if line[3:5] == country)\
    - pyo.quicksum(model_over_bounded.var_transmission[line,timestep] for line in model_over_bounded.set_line if line[0:2] == country)\
    + pyo.quicksum(model_over_bounded.var_tech_output[country,tech,timestep] for tech in dict_vary_tech_output[country,list_timestep[timestep][0]])\
    - pyo.quicksum(model_over_bounded.var_tech_input[country_tech,timestep] for country_tech in [country_tech for country_tech in list_country_tech_input if country_tech[0] == country])\
    == residual_demand.max()
model_over_bounded.constraint_electricity_balance = pyo.Constraint(model_over_bounded.set_country,model_over_bounded.set_timestep,rule=energy_balance_electricity)

def transmission_capacity_electricity(model_over_bounded,line,timestep):
    return model_over_bounded.var_transmission[line,timestep] <= df_line_capacity.loc[line,'Capacity']
model_over_bounded.constraint_transmission_capacity_electricity = pyo.Constraint(model_over_bounded.set_line,model_over_bounded.set_timestep,rule=transmission_capacity_electricity)

def tech_output_capacity(model_over_bounded,country,tech,timestep):
    if tech in ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River']:
        return model_over_bounded.var_tech_output[country,tech,timestep] <= dict_tech_profile[tech].loc[list_timestep[timestep],country].min()
    elif tech in ['Battery','Nuclear','Coal','CCGT','OCGT','Hydro_Storage']:
        name_capacity = tech
        return model_over_bounded.var_tech_output[country,tech,timestep] <= df_tech_capacity.loc[country,name_capacity]
    elif tech == 'E_ENS':
        return model_over_bounded.var_tech_output[country,tech,timestep] <= demand_profile[country].max()
    else:
        print('Error here!' + tech)
model_over_bounded.constraint_tech_output_capacity = pyo.Constraint(model_over_bounded.set_country_tech_timestep_output,\
                                                            rule=tech_output_capacity)

def tech_output_lower_bound(model_over_bounded,country,tech,timestep):
    if dict_vary_tech_output_lower_bound[country,tech,list_timestep[timestep][0]] > 0:
        if tech in ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River']:
            return model_over_bounded.var_tech_output[country,tech,timestep] >= min(dict_tech_profile[tech].loc[list_timestep[timestep],country].min(),dict_vary_tech_output_lower_bound[country,tech,list_timestep[timestep][0]]) 
        elif tech in ['Nuclear','Coal','CCGT','OCGT','E_ENS']: # 
            return model_over_bounded.var_tech_output[country,tech,timestep] >= dict_vary_tech_output_lower_bound[country,tech,list_timestep[timestep][0]]
        else:
            return pyo.Constraint.Skip
    else:
        return pyo.Constraint.Skip
model_over_bounded.constraint_tech_output_lower_bound = pyo.Constraint(model_over_bounded.set_country_tech_timestep_output,\
                                                            rule=tech_output_lower_bound)

def tech_input_capacity(model_over_bounded,country,tech,timestep):
    name_capacity = tech
    return model_over_bounded.var_tech_input[country,tech,timestep] <= df_tech_capacity.loc[country,name_capacity]

model_over_bounded.constraint_tech_input_capacity = pyo.Constraint(model_over_bounded.set_country_tech_input,model_over_bounded.set_timestep,\
                                                            rule=tech_input_capacity)

def stored_energy_capacity_uninvestable(model_over_bounded,country,tech,timestep):
    if tech == 'Battery':
        return model_over_bounded.var_stored_energy[country,tech,timestep] <= df_tech_capacity.loc[country,'Battery_MWh']
    else:
        return model_over_bounded.var_stored_energy[country,tech,timestep] <= 1000 * df_tech_capacity.loc[country,tech+'_GWh']
model_over_bounded.constraint_stored_energy_capacity_uninvestable = pyo.Constraint(model_over_bounded.set_country_tech_input,model_over_bounded.set_timestep,rule=stored_energy_capacity_uninvestable)

def storage_balance(model_over_bounded,country,tech,timestep):
    hours = list_timestep[timestep]
    eff = dict_storage_cycle_efficiency[tech]

    if WITH_STORAGE_DISSIPATION:
        diss = dict_storage_hourly_self_dissipate_rate[tech]
    else:
        diss = 0.0

    if timestep == 0:
        return  model_over_bounded.var_stored_energy[country,tech,timestep] - (1 - diss)**len(hours) * model_over_bounded.var_stored_energy[country,tech,len(list_timestep)-1] == len(list_timestep[timestep]) * (model_over_bounded.var_tech_input[country,tech,timestep] - (1/eff) * model_over_bounded.var_tech_output[country,tech,timestep])
    else:
        return  model_over_bounded.var_stored_energy[country,tech,timestep] - (1 - diss)**len(hours) * model_over_bounded.var_stored_energy[country,tech,timestep-1] == len(list_timestep[timestep]) * (model_over_bounded.var_tech_input[country,tech,timestep] - (1/eff) * model_over_bounded.var_tech_output[country,tech,timestep]) 
model_over_bounded.constraint_storage_balance_no_inflow = pyo.Constraint(model_over_bounded.set_country_tech_input,model_over_bounded.set_timestep,rule=storage_balance)

# -------- Objective --------
def total_cost_over(m):
    gen_cost = pyo.quicksum(
        len(list_timestep[t]) * dict_variable_cost[tech] *
        m.var_tech_output[country, tech, t]
        for (country, tech, t) in m.set_country_tech_timestep_output
    )

    if WITH_GRID_TARIFF:
        trans_cost = pyo.quicksum(
            df_grid_tarif.loc[line].values[0] *
            m.var_transmission[line, t]
            for line in m.set_line
            for t in m.set_timestep
        )
    else:
        trans_cost = 0

    return gen_cost + trans_cost
model_over_bounded.obj = pyo.Objective(rule=total_cost_over,sense=pyo.minimize)

solver = pyo.SolverFactory('gurobi')
model_over_bounded.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

mem_before = get_rss_GB()
solution_over = solver.solve(model_over_bounded,tee=True)
mem_after = get_rss_GB()


total_time_over = timeit.default_timer()-start_time
solver_time_over = solution_over['Solver'][0]['Wall time']
objective_over = solution_over['Problem'][0]['Lower bound']
over_delta_memory = mem_after - mem_before

In [ ]:
# ============================================================
# Extract Bounds from Overestimation
# ============================================================

dual_constraint_capacity_output_over = {}
output_tech_over = {}
for country,tech in list_country_tech_output:
    if tech in list_producer_tech:
        output_tech_over[country,tech] = []
        dual_constraint_capacity_output_over[country,tech] = []
        for timestep in range(len(list_timestep)):
            if tech in dict_vary_tech_output[country,list_timestep[timestep][0]]:
                output_tech_over[country,tech].append(model_over_bounded.var_tech_output[country,tech,timestep].value > 1e-5)
                dual_constraint_capacity_output_over[country,tech].append(min([dict_vary_tech_output_upper_bound[country,tech,hour] for hour in list_timestep[timestep]]) == model_over_bounded.var_tech_output[country,tech,timestep].value)
            else:
                output_tech_over[country,tech].append(True)
                dual_constraint_capacity_output_over[country,tech].append(True)

removed_hours_over = 0
for country,tech in list_country_tech_output:
    if tech in list_producer_tech:
        for timestep in range(N_CLUSTERS): # needs to be iterable later
            if tech in dict_vary_tech_output[country,list_timestep[timestep][0]]: 
                if dual_constraint_capacity_output_over[country,tech][timestep] == False:
                    if output_tech_over[country,tech][timestep] == False:
                        for hour in list_timestep[timestep]:
                            removed_hours_over += 1
                            dict_vary_tech_output[country,hour].remove(tech)
                    else:
                        for hour in list_timestep[timestep]:
                            dict_vary_tech_output_upper_bound[country,tech,hour] = model_over_bounded.var_tech_output[country,tech,timestep].value

reduced_variables_over = removed_hours_over

print(f"✅ Overestimation bounds extracted")
print(f"   Removed hours: {removed_hours_over:,}")

In [ ]:
# ============================================================
# Save Surrogate Results
# ============================================================

# -------- File paths --------
base_dir = f"surrogates_individual/{YEAR}/(s{int(WITH_STORAGE)}_t{int(WITH_TRANSMISSION)}_gt{int(WITH_GRID_TARIFF)}_sd{int(WITH_STORAGE_DISSIPATION)})"
os.makedirs(base_dir, exist_ok=True)

# Use N_CLUSTERS instead of count_timestep for clarity
filename_vary_tech = f"{base_dir}/dict_vary_tech_output_{N_CLUSTERS}.p"
filename_capacity_tech = f"{base_dir}/dict_capacity_tech_output_{N_CLUSTERS}.p"
filename_lower_bound = f"{base_dir}/dict_lower_bound_{N_CLUSTERS}.p"
filename_upper_bound = f"{base_dir}/dict_upper_bound_{N_CLUSTERS}.p"

# -------- Save dictionaries --------
print("💾 Saving surrogate results...")

with open(filename_vary_tech, 'wb') as f:
    pickle.dump(dict_vary_tech_output, f)

with open(filename_capacity_tech, 'wb') as f:
    pickle.dump(dict_capacity_tech_output, f)

with open(filename_lower_bound, 'wb') as f:
    pickle.dump(dict_vary_tech_output_lower_bound, f)

with open(filename_upper_bound, 'wb') as f:
    pickle.dump(dict_vary_tech_output_upper_bound, f)

print(f"Saved to: {base_dir}")

# -------- Performance summary --------
df_performance = pd.DataFrame({
    'N_Clusters': [N_CLUSTERS],
    'Time_Underestimation_s': [solver_time_under],
    'Time_Overestimation_s': [solver_time_over],
    'Total_Time_Underestimation_s': [total_time_under],
    'Total_Time_Overestimation_s': [total_time_over],
    'Reduced_Variables_Underestimation': [reduced_variables_under],
    'Reduced_Variables_Overestimation': [reduced_variables_over],
    'Objective_Underestimation': [objective_under],
    'Objective_Overestimation': [objective_over],
})

df_performance.index.name = 'Iteration'
df_performance.to_csv(f"{base_dir}/performance_surrogate_{N_CLUSTERS}.csv")

print(f"✅ Performance summary saved")

# -------- Boundary consistency check --------
print("🔍 Checking boundary consistency...")

inconsistent_bounds = []

for (country, tech, hour) in dict_vary_tech_output_lower_bound.keys():
    if tech in ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River']:
        lower = min(dict_tech_profile[tech].loc[hour,country],dict_vary_tech_output_lower_bound[country,tech,hour]) # dict_vary_tech_output_lower_bound[(country, tech, hour)]
        upper = dict_vary_tech_output_upper_bound[(country, tech, hour)]
        if upper < lower - 1e-6:  # Allow small numerical tolerance
            inconsistent_bounds.append((country, tech, hour, lower, upper))
    elif tech in ['Nuclear','Coal','CCGT','OCGT','E_ENS']:
        lower = dict_vary_tech_output_lower_bound[country,tech,hour] 
        upper = dict_vary_tech_output_upper_bound[country,tech,hour]
        if upper < lower - 1e-6:  # Allow small numerical tolerance
            inconsistent_bounds.append((country, tech, hour, lower, upper))

if inconsistent_bounds:
    print(f"⚠️ Found {len(inconsistent_bounds)} inconsistent bounds:")
    for country, tech, hour, lower, upper in inconsistent_bounds[:10]:  # Show first 10
        print(f"   {country} {tech} Hour {hour}: [{lower:.2f}, {upper:.2f}]")
    if len(inconsistent_bounds) > 10:
        print(f"   ... and {len(inconsistent_bounds) - 10} more")
else:
    print("✅ All bounds are consistent")

# -------- Memory usage summary --------
df_memory = pd.DataFrame({
    'N_Clusters': [N_CLUSTERS],
    'Memory_Underestimation_GB': [under_delta_memory],  # From previous cells
    'Memory_Overestimation_GB': [over_delta_memory],   # Adjust as needed
})

df_memory.to_csv(f"{base_dir}/memory_surrogate_{N_CLUSTERS}.csv")
print(f"✅ Memory usage saved")

# -------- Final summary --------
print("\n📊 Surrogate Summary")
print(f"   Resolution: {SURROGATE_RESOLUTION} ({N_CLUSTERS} clusters)")
print(f"   Total reduced variables (under): {reduced_variables_under}")
print(f"   Total reduced variables (over): {reduced_variables_over}")